# Identify analyses to rerun

Classify each country and analysis type in a completed job, then write `data/config/rerun_pairs.json` for `scripts/massive/jtrauer/rerun_countries.py`.

The rerun list is the union of two sets:

1. Analyses whose results are stale because the code has changed: both OxCGRT analyses for every country (H1 removed; independent also referenced to run start), and `no_scaling` for Oceania and Singapore (`beta` initialisation).
2. Combinations we would have used if they had finished, but did not: requested analyses that are `not run` or have `no log`. Skipped analyses (no scaler data) and analyses that were never requested are omitted.


In [1]:
import json
import pandas as pd

from emu_renewal.constants import (
    ANALYSIS_TYPES,
    DATA_PATH,
    FULL_RUN,
    OUTPUTS_PATH,
    OXCGRT_ANALYSIS_TYPES,
)
from emu_renewal.run import get_analyses_for_country
from emu_renewal.utils import get_cont_of_country

In [ ]:
run_id = FULL_RUN[0]

'59597639'

In [3]:
def classify_analysis(iso3, analysis, run_path, log_text):
    """Match run.py: complete if store_outputs finished, skipped if ScalerException."""
    if (run_path / iso3 / analysis / "updates.h5").exists():
        return "complete"
    if log_text is None:
        return "no log"
    if f"{analysis} data not available" in log_text:
        return "skipped"
    return "not run"


run_path = OUTPUTS_PATH / run_id
countries = json.load(open(DATA_PATH / "config/oxcgrt_included.json"))
status = pd.DataFrame(index=countries, columns=ANALYSIS_TYPES)
for iso3 in countries:
    log_path = run_path / iso3 / "run.log"
    log_text = log_path.read_text() if log_path.exists() else None
    requested = get_analyses_for_country(iso3)
    for analysis in ANALYSIS_TYPES:
        if analysis not in requested:
            status.loc[iso3, analysis] = "not requested"
        else:
            status.loc[iso3, analysis] = classify_analysis(iso3, analysis, run_path, log_text)

status.apply(pd.Series.value_counts).fillna(0).astype(int)

,no_scaling,oxcgrt_floored,oxcgrt_independent,g_mob,fb_visited_mob,fb_singletile_mob
complete,126,126,100,60,44,31
not requested,0,0,0,3,3,3
not run,0,0,26,49,66,79
skipped,0,0,0,14,13,13


In [4]:
def pairs_from_mask(mask):
    stacked = mask.stack()
    return list(stacked[stacked].index)

# All OxCGRT analyses changed because H1 removed from all (plus additional changes for independent)
changed = pd.DataFrame(False, index=status.index, columns=status.columns)
changed[OXCGRT_ANALYSIS_TYPES] = True

# Oceania analyses revised initialisation
oceania = [iso3 for iso3 in countries if get_cont_of_country(iso3) == "OC"]
changed.loc[oceania, "no_scaling"] = True
changed_pairs = pairs_from_mask(changed)

# Missing pairs
missing = status.isin(["not run", "no log"])
missing_pairs = pairs_from_mask(missing)

In [5]:
# Combine analyses that changed and runs that never went through
rerun_pairs = sorted(set(changed_pairs) | set(missing_pairs))
len(rerun_pairs)

449

In [6]:
json.dump(rerun_pairs, open(DATA_PATH / "config/rerun_pairs.json", "w"))